# Warm up Qwen3.5-4B for OpenEnv's REPL

This notebook trains the parent model on AlphaXiv demonstrations replayed in OpenEnv. Select **Runtime → Change runtime type → T4 GPU** and upload both JSONL splits produced by the converter. The default run performs one optimizer step on the longest training example.

In [ ]:
# This 4-bit LoRA example does not use Colab's optional TorchAO integration.
%pip uninstall -y torchao
# Keep Arrow within the range supported by Colab's bundled cuDF.
%pip install -q "trl[peft] @ git+https://github.com/huggingface/trl.git@99bbdfec75ad1a57bb2df9c72278b391508eda5f" "transformers==5.23.0" "datasets==5.0.0" "pyarrow>=21,<24" "bitsandbytes>=0.45.2"

## Load and inspect the replayed conversations

Run setup before importing training libraries. If you already ran a previous version, restart the session after setup, then resume here. The preflight checks source-paper separation, the assistant loss mask, and sequence lengths in both splits.

In [ ]:
from pathlib import Path

from datasets import load_dataset
from google.colab import files
from transformers import AutoTokenizer

from trl.chat_template_utils import get_training_chat_template


MODEL_ID = "Qwen/Qwen3.5-4B"
MODEL_REVISION = "851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a"
MAX_LENGTH = 8192
SMOKE_RUN = True
DATA_FILES = {split: Path(f"/content/laguna-alphaxiv-selective.{split}.jsonl") for split in ("train", "validation")}

missing = [path for path in DATA_FILES.values() if not path.is_file()]
if missing:
    print("Upload:", ", ".join(path.name for path in missing))
    uploaded = files.upload()
    for path in missing:
        if path.name not in uploaded:
            raise ValueError(f"Upload {path.name}.")
        path.write_bytes(uploaded[path.name])

dataset = load_dataset("json", data_files={split: str(path) for split, path in DATA_FILES.items()})
if any(len(dataset[split]) == 0 for split in DATA_FILES):
    raise ValueError("Both splits must contain conversations.")
if set(dataset["train"]["source_paper_id"]) & set(dataset["validation"]["source_paper_id"]):
    raise ValueError("A source-paper group occurs in both splits.")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
template = get_training_chat_template(tokenizer, require_prefix_preservation=False)

In [ ]:
from itertools import groupby


lengths = {}
for split, records in dataset.items():
    lengths[split] = []
    for index, record in enumerate(records):
        messages = record["messages"]
        tokens = tokenizer.apply_chat_template(
            messages,
            chat_template=template,
            tokenize=True,
            return_dict=True,
            return_assistant_tokens_mask=True,
        )
        ids = tokens["input_ids"]
        mask = tokens["assistant_masks"]
        assistant_turns = [message["content"] for message in messages if message["role"] == "assistant"]
        masked_spans = [
            tokenizer.decode([token_id for token_id, _ in run])
            for selected, run in groupby(zip(ids, mask, strict=True), key=lambda pair: pair[1])
            if selected
        ]
        if len(ids) > MAX_LENGTH:
            raise ValueError(f"Row {index} has {len(ids)} tokens and would be truncated at {MAX_LENGTH}.")
        if (
            not assistant_turns
            or len(masked_spans) != len(assistant_turns)
            or any(
                span.strip()
                .removeprefix("<think>\n\n</think>")
                .removeprefix("</think>")
                .removesuffix("<|im_end|>")
                .strip()
                != turn.strip()
                for span, turn in zip(masked_spans, assistant_turns, strict=True)
            )
        ):
            raise ValueError(f"Row {index} does not mask exactly the parent assistant turns.")
        lengths[split].append(len(ids))

    print(f"{split}: {len(records)} conversations, longest {max(lengths[split])} tokens")

smoke_index = max(range(len(dataset["train"])), key=lengths["train"].__getitem__)
print(f"Smoke row: {lengths['train'][smoke_index]} tokens")

## Run one optimizer step and save the adapter

The smoke run uses rank-8 LoRA with a frozen NF4 base model, FP16 compute, activation checkpointing, and batch size one. It measures training memory on the longest training row. Validation is tokenized and checked before training. When `SMOKE_RUN = False`, the notebook runs one full epoch and evaluates the validation split at its end.

In [ ]:
import gc
import math

import torch
from peft import LoraConfig
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

from trl import SFTConfig, SFTTrainer


if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime before running this cell.")
print(torch.cuda.get_device_name(0))
# A previous smoke run may still hold the model in an interactive session.
globals().pop("trainer", None)
globals().pop("model", None)
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    dtype=torch.float16,
    attn_implementation="sdpa",
    device_map={"": 0},
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    ),
)

config = SFTConfig(
    output_dir="/content/rlm-warmup",
    max_length=MAX_LENGTH,
    assistant_only_loss=True,
    packing=False,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    eval_strategy="no" if SMOKE_RUN else "epoch",
    prediction_loss_only=True,
    activation_checkpointing=True,
    fp16=True,
    bf16=False,
    learning_rate=1e-4,
    num_train_epochs=1,
    max_steps=1 if SMOKE_RUN else -1,
    logging_steps=1,
    logging_nan_inf_filter=False,
    save_strategy="no",
    report_to="none",
    seed=42,
)
trainer = SFTTrainer(
    model=model,
    args=config,
    train_dataset=dataset["train"].select([smoke_index]) if SMOKE_RUN else dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
    peft_config=LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],
        task_type="CAUSAL_LM",
    ),
)
# T4's FP16 gradient scaler cannot unscale BF16 adapter gradients.
with torch.no_grad():
    for param in trainer.model.parameters():
        if param.requires_grad:
            param.data = param.data.float()
result = trainer.train()
if not math.isfinite(result.training_loss) or (
    SMOKE_RUN and (result.global_step != 1 or trainer.accelerator.optimizer_step_was_skipped)
):
    raise RuntimeError(f"Unexpected training result: {result}")
print(f"Optimizer steps: {result.global_step}, loss: {result.training_loss:.6f}")
print(f"Peak allocated GPU memory: {torch.cuda.max_memory_allocated() / 2**30:.2f} GiB")
print(f"Peak reserved GPU memory: {torch.cuda.max_memory_reserved() / 2**30:.2f} GiB")
adapter_dir = "/content/rlm-warmup/adapter-4b-smoke" if SMOKE_RUN else "/content/rlm-warmup/adapter-4b"
trainer.save_model(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

In [ ]:
import shutil


archive = shutil.make_archive(adapter_dir, "zip", adapter_dir)
files.download(archive)